# Exploring the `llm` package

A hands-on tour of the project's LLM layer: what each piece does, what goes in and comes out, and how the pieces combine into research workflows.

| # | Section | What you'll see |
|---|---------|-----------------|
| 0 | Setup | Imports, run settings, and an **offline mode** for exploring without API keys |
| 1 | Map of the package | Which module does what |
| 2 | Providers | Choosing OpenAI / Featherless / others, and how defaults come from `.env` |
| 3 | Requests & responses | `ChatRequest` and `ChatResponse`, the objects everything passes around |
| 4 | Single calls | `generate()` and `chat()` |
| 5 | Concurrent inference | `generate_many()` over a population of tarot readings, with checkpointing and retries |
| 6 | Batch inference (OpenAI) | The JSONL file format, submitting, polling, and collecting results later |
| 7 | Validating outputs | Regex checks that every position has the drawn card, stored on `response.validation` *before* results are saved |
| 8 | Sentence embeddings | `embed()`, `embed_documents()`, similarity matrices, and embedding trajectories |
| 9 | White-box models (NNsight) | Hidden states, batched forward passes, and generation with per-step states |
| 10 | Workflow: black box vs white box | Comparing sentence-embedding geometry to each layer's hidden-state geometry |
| 11 | Extending | Adding a new provider (e.g. Anthropic) |

**Cost guide.** With `OFFLINE = True` nothing in this notebook costs anything. With `OFFLINE = False`:
- Sections 4, 5 and 8 make live calls to `PROVIDER`; section 7 makes 2.
- Section 6 only *submits* a real batch if `RUN_BATCH = True`.
- Section 9 runs locally on a small model unless `WHITEBOX_REMOTE = True`.

## 0. Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # torch/numpy OpenMP clash on Windows

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "sandbox" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import llm
from llm import (
    ChatRequest, ChatResponse, EmbeddingResponse, BatchJob, LLMProvider, OpenAICompatibleProvider,
    available_providers, get_provider, register_provider,
    chat, generate, generate_many,
    run_batch, wait_for_batch, run_batch_results, save_requests, load_requests, save_responses, load_responses,
    embed, embed_documents, embed_samples, split_into_sentences,
)

pd.set_option("display.max_colwidth", 120)
print("llm package loaded from", Path(llm.__file__).parent)

### Run settings

Change these, then re-run the notebook from here.

In [ ]:
OFFLINE = False                  # True: no network calls at all; uses the OfflineProvider defined below
PROVIDER = "featherless"         # live provider for chat + embeddings: "featherless" | "openai"
BATCH_PROVIDER = "openai"        # the Batch API is OpenAI-only
BATCH_MODEL = "gpt-6-sol"               # None -> OPENAI_MODEL from .env; or e.g. "gpt-4o-mini"
RUN_BATCH = False                # True actually submits a (small, paid) OpenAI batch in section 6

N_SAMPLES = 4                    # tarot readings to generate in section 5
MAX_TOKENS = 2000                # per reading
MAX_WORKERS = 2                  # concurrent requests (stay within your plan's concurrency limit)

WHITEBOX_REMOTE = False          # True: run on NDIF (needs NDIF_API_TOKEN); False: run a small model locally
WHITEBOX_MODEL = "openai-community/gpt2"   # remote example: "meta-llama/Llama-3.1-70B-Instruct"

OUT_DIR = REPO_ROOT / "data" / "llm_batches" / "explore"   # gitignored
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Offline provider

This is a complete provider in about 50 lines. It fakes chat, embeddings and batch jobs, so every cell runs without keys when `OFFLINE = True`. Its outputs are meaningless: replies are templated and embeddings are hash-based random vectors.

It is also the template for adding a real provider (see section 11). Subclass `LLMProvider`, implement `chat` (plus `embed` and the batch methods if the service has them), and call `register_provider`. Every function in the package then works with `provider="offline"`.

In [ ]:
import hashlib
import re
import time
import uuid
from llm.types import BATCH_COMPLETED, Usage


class OfflineProvider(LLMProvider):
    name = "offline"
    supports_embeddings = True
    supports_batch = True

    def __init__(self, dim: int = 64, **kwargs):
        kwargs.setdefault("default_model", "offline-echo")
        kwargs.setdefault("default_embedding_model", f"offline-hash-{dim}")
        super().__init__(**kwargs)
        self.dim = dim
        self._batches: dict[str, list[ChatRequest]] = {}

    def chat(self, request: ChatRequest) -> ChatResponse:
        time.sleep(0.05)  # pretend latency, so concurrency is visible
        # tarot prompts contain "Position: Card" lines; restate them the way real models do
        cards = re.findall(r"^([A-Z][\w /]+): (.+)$", request.prompt, flags=re.M)
        if cards:
            text = "Here is your reading.\n\n" + "\n\n".join(
                f"{i}. {pos}: {card} - this position points to meaningful change in your life."
                for i, (pos, card) in enumerate(cards, 1)) + "\n\nOverall, a spread about change."
        else:
            text = f"This is an offline reply to: {request.prompt[:80]}. It has a second sentence."
        n = len(text.split())
        return ChatResponse(request_id=request.id, text=text, model=self.resolve_model(request.model),
                            finish_reason="stop", usage=Usage(len(request.prompt.split()), n, n),
                            metadata=request.metadata)

    def embed(self, texts, model=None, **kwargs) -> EmbeddingResponse:
        vecs = []
        for t in texts:
            seed = int.from_bytes(hashlib.sha256(t.encode()).digest()[:8], "little")
            vecs.append(np.random.default_rng(seed).standard_normal(self.dim))
        return EmbeddingResponse(np.asarray(vecs, dtype=np.float32), model=self.resolve_embedding_model(model))

    # batch: completes instantly, results computed on collection
    def submit_batch(self, requests, **kwargs) -> BatchJob:
        batch_id = f"offline_batch_{uuid.uuid4().hex[:8]}"
        self._batches[batch_id] = list(requests)
        return self.get_batch(batch_id)

    def get_batch(self, batch_id) -> BatchJob:
        n = len(self._batches[batch_id])
        return BatchJob(id=batch_id, provider=self.name, status=BATCH_COMPLETED, raw_status="completed",
                        request_counts={"total": n, "completed": n, "failed": 0})

    def batch_results(self, job, requests=None):
        batch_id = job.id if isinstance(job, BatchJob) else job
        return [self.chat(r) for r in (requests or self._batches[batch_id])]


register_provider("offline", lambda **kw: OfflineProvider(**kw))

if OFFLINE:
    PROVIDER = BATCH_PROVIDER = "offline"
    RUN_BATCH = True  # free offline
print(f"chat/embeddings -> {PROVIDER!r}, batch -> {BATCH_PROVIDER!r}")

## 1. Map of the package

| Module | Main entry points | Input → output |
|---|---|---|
| `llm.types` | `ChatRequest`, `ChatResponse`, `EmbeddingResponse`, `BatchJob` | shared containers |
| `llm.providers` | `get_provider`, `register_provider`, `LLMProvider`, `OpenAICompatibleProvider` | name / `.env` → provider object |
| `llm.inference` | `generate`, `chat`, `generate_many` | prompt(s) / `ChatRequest`(s) → text / `ChatResponse`(s), live |
| `llm.batch` | `run_batch`, `wait_for_batch`, `run_batch_results`, `save_*` / `load_*` | `ChatRequest`s → `BatchJob` → `ChatResponse`s, asynchronous (≤24h, ~50% cheaper) |
| `llm.embeddings` | `embed`, `embed_documents`, `embed_samples`, `SentenceTransformerEmbedder` | text(s) → `np.ndarray` |
| `llm.whitebox` | `WhiteBoxModel` | text → `HiddenStates` / `GenerationTrace` (torch tensors) |
| `validation` *(separate package)* | `response_validator`, `celtic_cross_validator`, `summary_frame`, `issues_frame` | response text + expected input → `response.validation` dict |

Every function takes `provider=` as a name (`"openai"`, `"featherless"`), a provider object, or nothing (which uses `$LLM_PROVIDER`, else `"openai"`).

## 2. Providers

`get_provider(name)` builds a provider from `.env` and caches it, so the HTTP client is reused.

| Provider | Key | Chat model | Embedding model | Batch |
|---|---|---|---|---|
| `openai` | `OPENAI_API_KEY` / `OPENAI_API_TOKEN` | `OPENAI_MODEL` | `OPENAI_EMBEDDING_MODEL` (default `text-embedding-3-small`) | yes |
| `featherless` | `FEATHERLESS_API_TOKEN` | `FEATHERLESS_MODEL` | `EMBEDDING_MODEL` (default `Qwen/Qwen3-Embedding-8B`) | no |

In [ ]:
print("registered:", available_providers())

p = get_provider(PROVIDER)
print(p)
print("  default chat model:     ", p.default_model)
print("  default embedding model:", p.default_embedding_model)
print("  supports embeddings:    ", p.supports_embeddings)
print("  supports batch:         ", p.supports_batch)

Passing keyword overrides builds a **separate, uncached** provider. This is useful for a different default model or default sampling parameters without touching `.env`:

In [ ]:
creative = get_provider(PROVIDER, default_params={"temperature": 1.0, "top_p": 0.95})
print(creative.default_params, "| cached instance untouched:", get_provider(PROVIDER).default_params)

# provider-specific sampling defaults (sent via extra_body; see section 3)
featherless_sampling = get_provider(PROVIDER, default_extra_body={"top_k": 40, "min_p": 0.02})
print(featherless_sampling.default_extra_body)

## 3. Requests and responses

A **`ChatRequest`** has:
- `messages`: OpenAI-style role/content list
- `model`: `None` means the provider default
- `params`: `max_tokens`, `temperature`, …
- `id`: joins a response back to its request; becomes `custom_id` in batch files
- `metadata`: your own data, never sent to the model, copied onto the response

A **`ChatResponse`** has `text`, `model`, `finish_reason`, `usage`, `metadata`, and `error`. `error` is `None` on success; on failure `text` is `None` and `error` holds the message. The provider-native object is kept in `raw`.

In [ ]:
req = ChatRequest.from_prompt(
    "Name three uses of a magnifying glass.",
    system="You are concise.",
    id="demo-1",
    metadata={"topic": "optics"},
    max_tokens=100,
    temperature=0.3,
)
req

In [ ]:
# Multi-turn: build the message list yourself
convo = ChatRequest(messages=[
    {"role": "system", "content": "You are a tarot reader."},
    {"role": "user", "content": "What does the Tower mean?"},
    {"role": "assistant", "content": "Sudden upheaval, revelation, the collapse of false structures."},
    {"role": "user", "content": "And reversed?"},
], params={"max_tokens": 150})

print("last user message:", convo.prompt)
print("serialisable:", json.dumps(convo.to_dict())[:160], "...")

Params are written the same way for every provider. On OpenAI, `max_tokens` is renamed to `max_completion_tokens` automatically, because newer OpenAI models reject `max_tokens`. You can see the exact body a request turns into (not available for the offline provider):

In [ ]:
if isinstance(p, OpenAICompatibleProvider):
    print(json.dumps(p.build_chat_body(req), indent=2))
else:
    print("(offline provider: no HTTP body)")

### Provider-specific parameters: `extra_body`

Some servers accept sampling controls that aren't part of the standard OpenAI schema. Featherless (vLLM) takes `top_k`, `min_p` and `repetition_penalty`, for example. Put these in `extra_body`. They are sent verbatim at the top level of the request body, and in batch files they are merged into each line's `body`.

They can be set per request, per call (`generate(..., extra_body=...)`, `generate_many(prompts, extra_body=...)`), or for every request on a provider (`get_provider(name, default_extra_body=...)`). When sources overlap, request-level values win key by key.

**Caution:** OpenAI rejects unknown fields with a 400 error. Featherless silently ignores fields it doesn't recognise, so check the spelling.

In [ ]:
SAMPLING = {"repetition_penalty": 1.1, "top_k": 40, "min_p": 0.02}

sampled_req = ChatRequest.from_prompt("Name three uses of a magnifying glass.", max_tokens=100,
                                      extra_body={"top_k": 20})           # per-request value
sampled_llm = get_provider(PROVIDER, default_extra_body=SAMPLING)          # provider-wide defaults

print("merged extra_body:", sampled_llm.resolve_extra_body(sampled_req))  # top_k=20 overrides the default 40
if isinstance(sampled_llm, OpenAICompatibleProvider):
    print(json.dumps(sampled_llm.build_chat_body(sampled_req), indent=2))

## 4. Single calls

- `generate(prompt, ...)` returns a string, which is handy for quick checks.
- `chat(request, provider)` returns the full `ChatResponse`: usage, finish reason, metadata.

Both raise on API errors. `generate_many` (next section) is the error-tolerant option.

In [ ]:
sampling = {"repetition_penalty": 1.1, "top_k": 40, "min_p": 0.02}

text = generate(
    "Write one vivid sentence about a lighthouse keeper finding something strange after a storm.",
    system="You are a creative fiction writer.",
    provider=PROVIDER, max_tokens=120, temperature=0.9, extra_body=sampling
)
print(text)

In [ ]:
resp = chat(req, PROVIDER)
print("text:         ", resp.text)
print("model:        ", resp.model)
print("finish_reason:", resp.finish_reason, "  (\"length\" means max_tokens cut it off)")
print("usage:        ", resp.usage)
print("metadata:     ", resp.metadata, "  <- carried over from the request")

## 5. Concurrent inference: generating a population

`generate_many` fans requests out over a thread pool against the live endpoint:
- It returns responses **in input order**.
- A failed request becomes a response with `.error` set, and the rest keep going.
- `on_result` is called as each response lands. Use it to append to a JSONL checkpoint so a crash or kernel restart loses nothing.
- The SDK retries rate limits and 5xx errors with backoff. Keep `max_workers` within your plan's concurrency limit.

**Live vs batch:** live calls return in seconds at full price, while batch (section 6) returns within 24h at about half price. Use live calls for iteration and small populations, and batch for large ones.

Here we build the tarot population from the notebook's setup. Each request carries its `instant` (the card draw) as metadata.

In [ ]:
from coordinate_systems.state_instantiation import Instantiation

config = json.load(open(REPO_ROOT / "sandbox" / "vector_space_config.json"))
polarities = ["", " reversed"] if config.get("negative_polarity") else [""]
possible_states = [card + pol for card in config["aliases"] for pol in polarities]
step_aliases = config["step_aliases"]


def celtic_cross_prompt(instant: dict) -> str:
    cards = "\n".join(f"{pos}: {card}" for pos, card in instant.items())
    return ("You're a tarot reader. Can you read my Celtic cross tarot? Give an overview of the reading "
            f"after, any thoughts you have. I'll draw the cards:\n\n{cards}")


instants = [Instantiation(possible_states, step_aliases).instant for _ in range(N_SAMPLES)]
tarot_requests = [
    ChatRequest.from_prompt(celtic_cross_prompt(inst), id=f"tarot-{i}", metadata={"instant": inst},
                            max_tokens=MAX_TOKENS)
    for i, inst in enumerate(instants)
]
print(f"{len(possible_states)} possible states, {len(step_aliases)} positions, {len(tarot_requests)} requests\n")
print(tarot_requests[0].prompt)

In [ ]:
checkpoint = OUT_DIR / "tarot_live_responses.jsonl"
checkpoint.unlink(missing_ok=True)

responses = generate_many(
    tarot_requests,
    provider=PROVIDER,
    max_workers=MAX_WORKERS,
    on_result=lambda r: save_responses([r], checkpoint, append=True),   # checkpoint as they land
)
print(f"{sum(r.ok for r in responses)}/{len(responses)} succeeded; checkpoint: {checkpoint.name}")

In [ ]:
print(responses[0].text)
print(tarot_requests)
print(OUT_DIR)

Responses flatten naturally into a DataFrame. Metadata travels with each row, so the card draw sits next to the reading it produced.

In [ ]:
df = pd.DataFrame([{
    "id": r.request_id,
    "ok": r.ok,
    "present": r.metadata["instant"]["Present"],
    "outcome": r.metadata["instant"]["Outcome"],
    "finish": r.finish_reason,
    "completion_tokens": r.usage.completion_tokens if r.usage else None,
    "reading": r.text,
    "error": r.error,
} for r in responses])
print(df.columns)

**Retrying failures:** because responses line up with requests, re-running only the failed ones is a filter.

In [ ]:
failed = [req for req, resp in zip(tarot_requests, responses) if not resp.ok]
if failed:
    retried = {r.request_id: r for r in generate_many(failed, provider=PROVIDER, max_workers=MAX_WORKERS)}
    responses = [retried.get(r.request_id, r) for r in responses]
print(f"{len(failed)} retried; now {sum(r.ok for r in responses)}/{len(responses)} ok")

**Recovering from the checkpoint** (e.g. after a kernel restart). Rows arrive in completion order, so re-order them by id if needed.

In [ ]:
recovered = load_responses(checkpoint)
print(len(recovered), "responses recovered; first:", recovered[0].request_id, "->", (recovered[0].text or "")[:100], "...")

## 6. Batch inference (OpenAI Batch API)

Lifecycle:

```
ChatRequests ──build_batch_lines──▶ JSONL ──upload──▶ batch job ──(≤24h)──▶ output + error files ──▶ ChatResponses
                                                        status: pending → running → completed / failed / expired / cancelled
```

Each line of the input file is one request, and `custom_id` is the `ChatRequest.id`:

```json
{"custom_id": "tarot-0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "...", "messages": [...], "max_completion_tokens": 1200}}
```

Rules checked before upload: ids must be unique, the file can target **one model**, and the limits are ≤ 50,000 lines and ≤ 200 MB.

You can build and inspect the file for free, without submitting anything. If there's no OpenAI key, a key-less preview provider is used just to build lines.

In [ ]:
print("BATCH PROVIDER: ", BATCH_PROVIDER)
print("BATCH MODEL: ", BATCH_MODEL)
try:
    batch_llm = get_provider(BATCH_PROVIDER)
except ValueError as e:  # no OpenAI key configured
    print("No key for", BATCH_PROVIDER, "- using a preview provider (can build files, cannot submit).")
    batch_llm = OpenAICompatibleProvider(name="openai-preview", api_key="not-used",
                                         default_model=BATCH_MODEL or "gpt-4o-mini", supports_batch=True,
                                         param_renames={"max_tokens": "max_completion_tokens"})

batch_requests = [
    ChatRequest.from_prompt(celtic_cross_prompt(inst), id=f"tarot-batch-{i}", model=BATCH_MODEL,
                            metadata={"instant": inst}, max_tokens=MAX_TOKENS)
    for i, inst in enumerate(instants)
]
print(len(batch_requests))
print(batch_requests)

In [ ]:
if isinstance(batch_llm, OpenAICompatibleProvider):
    lines = batch_llm.build_batch_lines(batch_requests)
    preview = dict(lines[0], body={**lines[0]["body"], "messages": "[... omitted ...]"})
    print(json.dumps(preview, indent=2))

    preview_path = batch_llm.write_batch_file(lines, OUT_DIR / "tarot_preview_input.jsonl")
    print(f"\nwrote {len(lines)} lines, {preview_path.stat().st_size:,} bytes -> {preview_path.name}")
else:
    print("(offline provider: no JSONL file; the lifecycle below still works)")

The rules are enforced before anything is uploaded:

In [ ]:
if isinstance(batch_llm, OpenAICompatibleProvider):
    bad_batches = {
        "duplicate ids": [ChatRequest.from_prompt("a", id="x"), ChatRequest.from_prompt("b", id="x")],
        "mixed models": [ChatRequest.from_prompt("a", model="model-a"), ChatRequest.from_prompt("b", model="model-b")],
    }
    for label, bad in bad_batches.items():
        try:
            batch_llm.build_batch_lines(bad)
        except ValueError as e:
            print(f"{label:>14}: {e}")

### Submitting, then collecting later

The usual pattern is **submit now, collect later**, possibly from a different session:

1. Save the requests. They hold metadata that the batch API never sees.
2. Submit with `wait=False` and note the job id.
3. Later, check the status, then collect results joined back to the saved requests.

This cell only submits when `RUN_BATCH = True`, or in offline mode.

In [ ]:
job = None
requests_path = OUT_DIR / "tarot_batch_requests.jsonl"
RUN_BATCH=True
if RUN_BATCH:
    save_requests(batch_requests, requests_path)
    job = run_batch(batch_requests, provider=BATCH_PROVIDER, wait=False, metadata={"experiment": "llm_explore"})
    (OUT_DIR / "tarot_batch_job.json").write_text(json.dumps(job.to_dict(), indent=2))
    print("status:", job.status, "| job saved to tarot_batch_job.json")
else:
    print("RUN_BATCH is False - not submitting. Set it to True to run a real (paid) batch.")

In [ ]:
# Check on it at any time (cheap). A real OpenAI batch takes minutes to hours.
print(job.id)
if job is not None:
    current = get_provider(BATCH_PROVIDER).get_batch(job.id)
    print(current.status, current.request_counts)

In [ ]:
# Block until finished (poll_interval in seconds), then collect results in request order with metadata.
if job is not None:
    wait_for_batch(job, provider=BATCH_PROVIDER, poll_interval=60)
    batch_responses = run_batch_results(job, provider=BATCH_PROVIDER, requests=load_requests(requests_path))
    for r in batch_responses:
        print(r.request_id, "|", r.metadata["instant"]["Outcome"], "|", (r.text or r.error)[:90], "...")

In [ ]:
for r in batch_responses:
    print(r.request_id, r.text)
    print("\n--****--****--****--****--****\n")

**Resuming in a new session**, from only the saved files:

```python
job_id = json.loads((OUT_DIR / "tarot_batch_job.json").read_text())["id"]
responses = run_batch_results(job_id, provider="openai", requests=load_requests(OUT_DIR / "tarot_batch_requests.jsonl"))
```

For an expired batch, the requests that finished are returned normally. The rest come back with `error="batch_expired: ..."`, so filter on `.ok` and resubmit those.

**Embedding batches** work the same way, with a `{custom_id: text}` mapping:

In [ ]:
if isinstance(batch_llm, OpenAICompatibleProvider):
    emb_lines = batch_llm.build_embedding_batch_lines({"s1": "An insect is small.", "s2": "Lenses magnify."})
    print(json.dumps(emb_lines[0]))
# live: job = get_provider("openai").submit_embedding_batch({...}); vectors, errors = provider.embedding_batch_results(job)

## 7. Validating outputs (no extra LLM calls)

The `validation` package checks responses with regex and vocabulary matching only. For tarot readings, `celtic_cross_validator()` checks:

| Check | Fails (error) when | Warns when |
|---|---|---|
| `positions` | a position is missing (`missing_label`), listed without a card (`missing_value`), has the wrong card (`wrong_card`), or the wrong orientation (`orientation_mismatch`) | a position is restated with a different card; positions are out of order |
| `foreign_cards` | – | a card is mentioned that wasn't drawn |
| `not_truncated` | `finish_reason == "length"` | – |
| `no_refusal` | "I can't help…", "As an AI…" | – |
| `non_empty` | fewer than 200 characters | – |

**Where results go.** The verdict is stored on the response as `response.validation`, a JSON dict, with `response.valid` as a shortcut: `True`, `False`, or `None` if not validated.
- It is saved and reloaded with the response.
- It is separate from `response.ok`: `ok=True, valid=False` means the API call worked but the content is wrong.
- It is also separate from `metadata`, which holds the request's inputs.

**When it runs.** Pass `validate=` to `generate_many` / `run_batch` / `run_batch_results`, and each response is validated *before* `on_result` runs or you save it. So checkpoint files already contain the verdict.

**Retries (on by default).** With a validator, a response that fails validation is regenerated up to `validation_retries=2` more times. `response.attempts` counts the generations made, and `response.failed_attempts` keeps each rejected attempt's text and validation.

In [ ]:
from llm import apply_validation
from validation import Check, RegexCheck, Validator, WARNING, issues_frame, response_validator, result_of, summary_frame
from validation.tarot import TAROT_CARDS, celtic_cross_validator, spread_validator

cc = celtic_cross_validator()
cc

### Validating one reading directly

`Validator.validate(text, expected=..., finish_reason=...)` works on plain strings; no `llm` objects are needed. Here `expected` is the card draw that went into the prompt.

The `positions` check records what it extracted, and how: `line` means the position started its own line, and `inline` means it was found in prose.

In [ ]:
r0 = next(r for r in responses if r.ok)
res = cc.validate(r0.text, expected=r0.metadata["instant"], finish_reason=r0.finish_reason)
print(res.summary())

pos = res.checks["positions"].details
pd.DataFrame({"expected": r0.metadata["instant"], "extracted": pos["extracted"], "found via": pos["modes"]})

### What failures look like

Each variant below breaks the reading above in one realistic way. The last one mimics a real Featherless reply, where the model asked you to draw the cards instead of reading them.

In [ ]:
inst = r0.metadata["instant"]
text = r0.text
not_drawn = next(c for c in TAROT_CARDS if c not in {v.removesuffix(" reversed") for v in inst.values()})
outcome_card = inst["Outcome"].removesuffix(" reversed")
reversed_pos = next((p for p, c in inst.items() if c.endswith(" reversed")), None)

variants = {"original": (text, r0.finish_reason)}
if reversed_pos:
    card = inst[reversed_pos].removesuffix(" reversed")
    variants[f"'reversed' dropped ({reversed_pos})"] = (
        re.sub(re.escape(card) + r"\s*\(?reversed\)?", card, text, flags=re.I), "stop")
variants["wrong Outcome card"] = (text.replace(outcome_card, not_drawn), "stop")
cut = text.lower().find("outcome")
variants["cut off before Outcome"] = (text[:cut] if cut > 0 else text[: len(text) // 2], "length")
variants["refusal"] = ("I'm sorry, but I can't help with tarot readings. " * 5, "stop")
variants["asked user to draw cards"] = (
    "Certainly! Please draw your ten cards in the following order:\n" +
    "\n".join(f"{i}. {p}" for i, p in enumerate(inst, 1)) +
    "\n\nOnce you have drawn all the cards, I will proceed with the reading and share my thoughts.", "stop")

rows = []
for name, (t, fr) in variants.items():
    v = cc.validate(t, expected=inst, finish_reason=fr)
    rows.append({"variant": name, "passed": v.passed,
                 "errors": ", ".join(sorted({f"{i.code}" + (f" ({i.label})" if i.label else "") for i in v.errors})),
                 "warnings": ", ".join(sorted({i.code for i in v.warnings}))})
pd.DataFrame(rows)

### Validating during generation, before anything is written

`response_validator(validator, expected="instant")` turns a validator into the `validate=` hook. `expected="instant"` means "compare against `response.metadata["instant"]`". It can also be a dotted path, or a function `(response, request) -> expected`.

Validation runs in each worker thread before `on_result`, so the checkpoint file below already contains every verdict. Readings that fail validation are regenerated automatically, up to 2 more times each. *This cell makes 2 live requests, plus up to 4 retries.*

In [ ]:
validate = response_validator(cc, expected="instant")

validated_checkpoint = OUT_DIR / "tarot_validated_responses.jsonl"
validated_checkpoint.unlink(missing_ok=True)

validated = generate_many(
    tarot_requests[:2], provider=PROVIDER, max_workers=MAX_WORKERS,
    validate=validate,                                                         # runs first ...
    validation_retries=2,                                                      # (the default)
    on_result=lambda r: save_responses([r], validated_checkpoint, append=True),  # ... then this writes it
)
print([(r.request_id, "ok" if r.ok else "API error", r.valid, f"{r.attempts} attempt(s)") for r in validated])

on_disk = load_responses(validated_checkpoint)[0]
print("\nwritten to disk with:", {k: on_disk.validation[k] for k in ("passed", "validator", "n_errors", "n_warnings")})

### What a retry looks like

To show retries without spending anything, this cell uses a tiny scripted provider. For each prompt it returns a fixed sequence of replies. The first request fails validation twice before a good reading arrives. The second never produces a valid reading, so it is returned invalid after the default 2 retries, with its history attached.

In [ ]:
class ScriptedProvider(LLMProvider):
    """Returns replies[prompt][n] on the n-th call (last one repeats); no API calls."""
    name = "scripted"

    def __init__(self, replies):
        super().__init__(default_model="scripted")
        self.replies, self.calls = replies, {}

    def chat(self, request):
        n = self.calls[request.prompt] = self.calls.get(request.prompt, 0) + 1
        text = self.replies[request.prompt][min(n, len(self.replies[request.prompt])) - 1]
        return ChatResponse(request.id, text, finish_reason="stop", metadata=request.metadata)


good = r0.text
wrong_outcome = good.replace(outcome_card, not_drawn)
scripted = ScriptedProvider({
    "eventually good": [wrong_outcome, variants["asked user to draw cards"][0], good],
    "never good": [wrong_outcome],
})
demo = [ChatRequest.from_prompt(p, id=p, metadata={"instant": inst}) for p in scripted.replies]
retried = generate_many(demo, provider=scripted, validate=validate, max_workers=1)

for r in retried:
    print(f"\n{r.request_id}: valid={r.valid} after {r.attempts} attempt(s)")
    for k, f in enumerate(r.failed_attempts, 1):
        codes = sorted({i["code"] for i in f["validation"]["issues"] if i["severity"] == "error"})
        print(f"  rejected attempt {k}: {codes}")

**Batch results** take the same hook:

```python
responses = run_batch_results(job_id, provider="openai", requests=load_requests(...), validate=validate)
```

For responses you already have, including files saved before validation existed, use `apply_validation`. It validates in place and returns the list:

In [ ]:
apply_validation(responses, validate)                      # section 5's live readings
batch_validated = []
if job is not None and "batch_responses" in globals():
    batch_validated = apply_validation(batch_responses, validate)

# re-validating a saved file, e.g. after improving a rule
from_disk = apply_validation(load_responses(checkpoint), validate)
save_responses(from_disk, OUT_DIR / "tarot_live_responses_validated.jsonl")

all_validated = responses + validated + batch_validated
summary_frame(all_validated)

### Analysing issues across a population

`issues_frame` gives one row per issue. Grouping by `code` and `label` shows what a model systematically gets wrong, e.g. a particular position, or dropping "reversed".

In [ ]:
issues = issues_frame(all_validated)
if issues.empty:
    print("No issues in", len(all_validated), "responses")
else:
    display(issues.groupby(["severity", "check", "code"]).size().rename("count").to_frame())
    display(issues[issues.label.notna()].groupby(["label", "code"]).size().rename("count").to_frame())

In [ ]:
# drill into any failing response: result_of() rebuilds the full ValidationResult from the stored dict
for r in all_validated:
    if r.valid is False:
        print(f"--- {r.request_id}")
        print(result_of(r).summary())

### Customising: extra rules, other spreads, other tasks

A `Validator` is just a list of checks. You can:
- add rules to the tarot validator;
- reuse it for other spreads with `spread_validator`;
- write your own `Check` subclass: implement `run(ctx)` and report findings with `self.issue(code, message, severity)`.

In [ ]:
class WordCount(Check):
    name = "word_count"

    def __init__(self, lo, hi):
        self.lo, self.hi = lo, hi

    def run(self, ctx):
        n = len(ctx.text.split())
        issues = [] if self.lo <= n <= self.hi else [
            self.issue("word_count", f"{n} words (wanted {self.lo}-{self.hi})", WARNING)]
        return self.result(issues, n_words=n)


strict = Validator([
    *cc.checks,
    RegexCheck("has_overview", required={"overview": r"\b(overall|in summary|overview)\b"}, severity=WARNING),
    WordCount(150, 900),
], name="celtic_cross_strict")
print(strict.validate(r0.text, expected=r0.metadata["instant"], finish_reason=r0.finish_reason).summary())

three_card = spread_validator(["Past", "Present", "Future"], min_chars=20)
draw = {"Past": "The Fool", "Present": "Two of Cups reversed", "Future": "The Sun"}
reading = "Past: The Fool - a leap.\nPresent: Two of Cups - harmony.\nFuture: The Sun - joy."   # deliberately drops 'reversed'
print("\nthree-card spread:\n" + three_card.validate(reading, expected=draw).summary())

## 8. Sentence embeddings

- `embed(str)` returns shape `(dim,)`, and `embed(list)` returns `(n, dim)` in input order.
- Texts are sent `batch_size` per call (default 256), and `max_workers > 1` runs calls in parallel.
- `normalize=True` gives unit vectors, so dot product equals cosine similarity.
- Extra kwargs pass through to the API (e.g. `dimensions=256` for OpenAI `text-embedding-3-*`).

*In offline mode the vectors are random, so the plots show structure only by chance.*

In [ ]:
v = embed("A magnifying glass makes small things look bigger.", provider=PROVIDER)
print("single text ->", v.shape, "| norm:", round(float(np.linalg.norm(v)), 4))

In [ ]:
statements = {
    "premise_1": "An insect usually has a small size.",
    "premise_2": "A magnifying glass is used to see small things by making objects appear bigger.",
    "conclusion": "Therefore, a magnifying glass can be used to see an insect by making it appear bigger.",
    "paraphrase": "Looking at an insect through a magnifying glass makes it appear larger.",
    "negated": "A magnifying glass cannot be used to see an insect by making it appear bigger.",
    "swapped": "A magnifying glass can be used to see an insect by making it appear smaller.",
    "unrelated": "The stock market closed higher on Tuesday.",
}
labels, sentences = list(statements), list(statements.values())

E = embed(sentences, provider=PROVIDER, normalize=True)
sim = pd.DataFrame(E @ E.T, index=labels, columns=labels)

plt.figure(figsize=(7, 5.5))
sns.heatmap(sim, annot=True, fmt=".2f", cmap="viridis", square=True)
plt.title("Cosine similarity of sentence embeddings")
plt.tight_layout()

### Embedding documents chunk by chunk

`embed_documents` splits each document with `splitter` (default: sentences) and embeds all chunks from all documents together in shared API calls. That is far fewer requests than one call per chunk. It returns `{key: {"chunks": [...], "embeddings": (n_chunks, dim)}}`.

Here each tarot reading becomes a **trajectory** through embedding space, one point per sentence. The step sizes between consecutive sentences give a simple "velocity" signal: large steps are topic shifts.

In [ ]:
readings = {r.request_id: r.text for r in responses if r.ok}
docs = embed_documents(readings, provider=PROVIDER, normalize=True)

rows = []
for key, d in docs.items():
    steps = np.linalg.norm(np.diff(d["embeddings"], axis=0), axis=1)   # ||e[t+1] - e[t]||
    rows.append({"id": key, "n_chunks": len(d["chunks"]), "mean_step": steps.mean() if len(steps) else np.nan,
                 "max_step": steps.max() if len(steps) else np.nan})
    plt.plot(steps, marker=".", label=key)

plt.xlabel("sentence index"); plt.ylabel("step size (chord distance)")
plt.title("Embedding 'velocity' along each reading"); plt.legend(fontsize=8); plt.tight_layout()
pd.DataFrame(rows)

In [ ]:
# the chunks behind the largest jump in the first reading
key = next(iter(docs))
chunks, emb = docs[key]["chunks"], docs[key]["embeddings"]
if len(chunks) > 1:
    j = int(np.argmax(np.linalg.norm(np.diff(emb, axis=0), axis=1)))
    print(f"[{key}] biggest shift between sentence {j} and {j + 1}:\n  - {chunks[j]}\n  - {chunks[j + 1]}")

**Custom splitters:** any `str -> list[str]` function works, e.g. fixed word windows, or groupings from `text_tokenizers` / `DynamicalEmbedding`. `splitter=None` embeds each document whole.

In [ ]:
def word_windows(text, size=25, stride=15):
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - size, 0) + 1, stride)] or [text]

windowed = embed_documents(readings, splitter=word_windows, provider=PROVIDER)
whole = embed_documents(readings, splitter=None, provider=PROVIDER)
k = next(iter(readings))
print("sentence chunks:", len(docs[k]["chunks"]), "| 25-word windows:", len(windowed[k]["chunks"]),
      "| whole doc:", whole[k]["embeddings"].shape)

`embed_samples` keeps the old notebook's in-place behaviour for a `samples` dict:

In [ ]:
samples = {i: {"instant": r.metadata["instant"], "reading": r.text} for i, r in enumerate(responses) if r.ok}
samples = embed_samples(samples, text_key="reading", provider=PROVIDER)
print(list(samples[0].keys()), samples[0]["embeddings"].shape)

**Local embeddings** (no API): pass a `SentenceTransformerEmbedder` as `provider`. This needs `pip install sentence-transformers`.

In [ ]:
import importlib.util

if importlib.util.find_spec("sentence_transformers"):
    from llm import SentenceTransformerEmbedder
    local = SentenceTransformerEmbedder("all-MiniLM-L6-v2")
    print(embed(sentences, provider=local).shape)
else:
    print("sentence-transformers not installed; skipping")

## 9. White-box models (NNsight)

`WhiteBoxModel` wraps an NNsight `LanguageModel`. With `remote=True` it runs on NDIF, so the weights are never downloaded. With `remote=False` it runs locally.

| Method | Returns | Shapes |
|---|---|---|
| `hidden_states(text, layers=None)` | `HiddenStates` | `.hidden_states`: `(n_layers, seq_len, hidden)` |
| `hidden_states_many(texts, batch_size=8)` | `list[HiddenStates]` | as above, each at its own true length |
| `generate(prompt, max_new_tokens)` | `GenerationTrace` | `.step_hidden_states[0]`: `(n_layers, prompt_len, hidden)`, later steps `(n_layers, 1, hidden)` |

`HiddenStates` also carries `tokens`, `token_ids`, `next_token`, and optional `next_token_logits`, plus the helpers `.mean_pool()`, `.last_token()` and `.layer(i)`.

This section defaults to GPT-2 locally: free, runs on CPU, and downloads about 500 MB once. For the real thing, set `WHITEBOX_REMOTE = True` and `WHITEBOX_MODEL = "meta-llama/Llama-3.1-70B-Instruct"`. Each trace is then one NDIF request, and all-layer states for a 70B model are large (80 × seq × 8192).

In [ ]:
from llm import WhiteBoxModel

wb = WhiteBoxModel(WHITEBOX_MODEL, remote=WHITEBOX_REMOTE, **({} if WHITEBOX_REMOTE else {"device_map": "cpu"}))
print(wb)

In [ ]:
hs = wb.hidden_states(statements["premise_1"], save_logits=True)
print("hidden_states:", tuple(hs.hidden_states.shape), "  (layers, tokens, hidden)")
print("tokens:       ", hs.tokens)
print("next token:   ", repr(hs.next_token))

top = hs.next_token_logits.float().softmax(-1).topk(5)
print("top-5 next:   ", [(wb.tokenizer.decode([int(i)]), round(float(p), 3)) for p, i in zip(top.values, top.indices)])

In [ ]:
# how the residual stream grows through the network
norms = hs.hidden_states.float().norm(dim=-1)          # (layers, tokens)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(norms.mean(dim=1), label="mean over tokens")
axes[0].plot(norms[:, -1], label="last token")
axes[0].set(xlabel="layer", ylabel="L2 norm", title="Hidden-state norm by layer"); axes[0].legend()
sns.heatmap(norms.T.numpy(), ax=axes[1], yticklabels=hs.tokens, cmap="magma")
axes[1].set(xlabel="layer", title="Norm per token per layer")
plt.tight_layout()

**Watch the first token.** Its norm is typically orders of magnitude larger than every other position from the early layers on. This is the "attention sink" or massive-activation effect; in Llama the first token is `<|begin_of_text|>`. A plain `.mean_pool()` is therefore dominated by one token that carries almost no content, which pushes every pair of texts toward about 0.99 cosine similarity. For comparing texts, pool over the positions *after* the first:

In [ ]:
def content_pool(h):
    # (layers, hidden): mean over tokens, skipping the first (sink) position when there is more than one
    states = h.hidden_states.float()
    return states[:, 1:].mean(dim=1) if states.shape[1] > 1 else states.mean(dim=1)

print("first-token norm / median other-token norm, per layer:")
print((norms[:, 0] / norms[:, 1:].median(dim=1).values).round().int().tolist())

### Batched forward passes

`hidden_states_many` sends `batch_size` texts per trace (one NDIF request each, instead of one per text). Mixed lengths are fine: batches are right-padded and each result is cut back to its own tokens, so results match single calls exactly.

`layers=` keeps only the layers you need, which cuts the download proportionally.

In [ ]:
many = wb.hidden_states_many(sentences, batch_size=8)
print([tuple(h.hidden_states.shape) for h in many])

every_4th = wb.hidden_states_many(sentences[:2], layers=range(0, wb.n_layers, 4), progress=False)
print("layer subset:", every_4th[0].layers, tuple(every_4th[0].hidden_states.shape))

**Where in the network does meaning separate?** Take the content-pooled state of each statement at every layer, and track its cosine similarity to the conclusion. Paraphrases should stay close; negations and swaps may only separate in certain layers. The dashed lines show the naive `.mean_pool()` for comparison: the first token flattens everything toward 1.

In [ ]:
import torch

pooled = torch.stack([content_pool(h) for h in many])              # (n_statements, layers, hidden)
naive = torch.stack([h.mean_pool().float() for h in many])
i_ref = labels.index("conclusion")
cos = torch.nn.functional.cosine_similarity(pooled, pooled[i_ref].unsqueeze(0), dim=-1)      # (n, layers)
cos_naive = torch.nn.functional.cosine_similarity(naive, naive[i_ref].unsqueeze(0), dim=-1)

plt.figure(figsize=(9, 4.5))
for k, lab in enumerate(labels):
    if lab != "conclusion":
        line, = plt.plot(cos[k].numpy(), label=lab)
        plt.plot(cos_naive[k].numpy(), ls="--", lw=0.8, color=line.get_color())
plt.xlabel("layer"); plt.ylabel("cosine similarity to conclusion"); plt.legend(fontsize=8)
plt.title("Hidden-state similarity by layer (solid: skip first token, dashed: naive mean)"); plt.tight_layout()

### Generation with per-step hidden states

- `generate` records every requested layer at every decoding step.
- `last_token_only=True` keeps one position per step, which is much smaller for long prompts.
- `.new_token_states()` returns `(layers, n_new, hidden)`: the state that *predicted* each new token.
- Instruct models expect `format_chat()` (the model's chat template); GPT-2 has no template, so it gets the raw prompt.

In [ ]:
question = "The lighthouse keeper opened the box and found"
prompt = wb.format_chat(question) if getattr(wb.tokenizer, "chat_template", None) else question

g = wb.generate(prompt, max_new_tokens=20, do_sample=False, last_token_only=True)
print(repr(g.text))
print("new-token states:", tuple(g.new_token_states().shape))

plt.figure(figsize=(10, 3.5))
sns.heatmap(g.new_token_states().float().norm(dim=-1).numpy(), cmap="magma",
            xticklabels=[wb.tokenizer.decode([t]) for t in g.new_token_ids])
plt.xlabel("generated token"); plt.ylabel("layer"); plt.title("Hidden-state norm while generating")
plt.tight_layout()

## 10. Workflow: black box vs white box

The two halves of the package combine naturally. The same statements go through the sentence-embedding model (section 8) and the white-box model (section 9), and the question is: **at which layer does the LLM's internal geometry most resemble the sentence-embedding geometry?**

For each layer, compute the statement-by-statement similarity matrix of the content-pooled hidden states. Then correlate its off-diagonal entries with those of the embedding similarity matrix. This is a simple form of representational similarity analysis.

*Only meaningful with `OFFLINE = False`, because offline embeddings are random.*

In [ ]:
iu = np.triu_indices(len(labels), k=1)
emb_sim = (E @ E.T)[iu]

normed = torch.nn.functional.normalize(pooled, dim=-1)                   # (n, layers, hidden)
layer_sims = torch.einsum("ilh,jlh->lij", normed, normed).numpy()        # (layers, n, n)
rsa = [np.corrcoef(emb_sim, s[iu])[0, 1] for s in layer_sims]

plt.figure(figsize=(8, 3.5))
plt.plot(rsa, marker="o")
plt.xlabel("layer"); plt.ylabel("Pearson r with embedding similarities")
plt.title(f"Representational similarity: {wb.model_name} layers vs sentence embeddings")
plt.tight_layout()
print("best-matching layer:", int(np.nanargmax(rsa)))

### Other workflow recipes

**Population → analysis (live, small N):**
`Instantiation` → `ChatRequest(metadata=instant)` → `generate_many(on_result=checkpoint)` → `embed_documents` → trajectories / DataFrame.

**Population at scale (batch, large N):**
build requests → `save_requests` → `run_batch(wait=False)` → save `job.id` → (later) `run_batch_results(job_id, requests=load_requests(...))` → filter `.ok`, resubmit failures.

**LLM-assisted annotation** (e.g. the entity / coreference prompts from `population_generator`): format a prompt per reading, `generate_many(..., temperature=0)`, then `json.loads(r.text)` for each response, with a `try/except` that records parse failures on the row.

**Interpretability on generated text:**
readings from `generate_many` → `wb.hidden_states_many(texts, layers=...)` → per-layer statistics (variance, differences between consecutive layers, similarity to reference statements).

## 11. Extending: adding a provider

`OfflineProvider` in section 0 is the full pattern. A real Anthropic provider would look like this:

```python
from anthropic import Anthropic
from llm import LLMProvider, ChatResponse, register_provider
from llm.types import Usage

class AnthropicProvider(LLMProvider):
    name = "anthropic"
    supports_batch = True            # Message Batches API

    def __init__(self, api_key=None, **kw):
        super().__init__(**kw)
        self.client = Anthropic(api_key=api_key)

    def chat(self, request):
        system = " ".join(m["content"] for m in request.messages if m["role"] == "system")
        messages = [m for m in request.messages if m["role"] != "system"]
        params = {"max_tokens": 1024, **self.default_params, **request.params}   # max_tokens is required
        msg = self.client.messages.create(model=self.resolve_model(request.model), system=system or None,
                                          messages=messages, **params)
        return ChatResponse(request.id, msg.content[0].text, model=msg.model, finish_reason=msg.stop_reason,
                            usage=Usage(msg.usage.input_tokens, msg.usage.output_tokens,
                                        msg.usage.input_tokens + msg.usage.output_tokens),
                            metadata=request.metadata, raw=msg)

    # submit_batch / get_batch / batch_results would wrap client.messages.batches.*,
    # using ChatRequest.id as each request's custom_id.

register_provider("anthropic", lambda **kw: AnthropicProvider(**kw))
```

After that, `generate_many(reqs, provider="anthropic")`, `run_batch(..., provider="anthropic")` and the rest work unchanged.